# Clasificación de Sectores de Patentes con TF-IDF + Regresión Logística

**Objetivo:** Entrenar un clasificador sobre `ipc_sector.csv` y aplicarlo al nuevo dataset de la OMPI.

**Variable objetivo:** `Sector` (5 clases: Química, Ingeniería mecánica, Instrumentos, Otros sectores, Electricidad - Electrónica)

**Features:** dos bloques de texto combinados
1. Códigos IPC (`Clasificacion` en entrenamiento / `IPC` en aplicación)
2. Título de la patente **en inglés** (`Titulo_en` / `Title`)

### Por qué se traduce el título

El dataset de entrenamiento tiene los títulos en español; el dataset al que se
aplica el modelo los tiene en inglés. Si se entrenara con títulos en español,
el vocabulario aprendido (*"composición"*, *"dispositivo"*) no aparecería nunca
en los datos nuevos (*"composition"*, *"device"*) y esa parte del modelo
aportaría cero. Traducir el entrenamiento al idioma de la aplicación es lo que
hace que el título sea utilizable como feature.

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight

# Compatibilidad entre versiones de scikit-learn: el argumento multi_class de
# LogisticRegression quedó obsoleto en 1.5 y fue ELIMINADO en 1.7. Si el
# notebook se corre en una versión nueva con multi_class="ovr" escrito a mano,
# revienta con TypeError. Esto lo pasa solo si la versión instalada lo acepta.
import inspect
PARAMS_LR = dict(class_weight="balanced", max_iter=1000,
                 random_state=42, C=1.0, solver="lbfgs")
if "multi_class" in inspect.signature(LogisticRegression).parameters:
    PARAMS_LR["multi_class"] = "ovr"      # one-vs-rest para 5 clases
else:
    print("scikit-learn >= 1.7: multi_class ya no existe, se usa multinomial "
          "(el comportamiento por defecto, y en general algo mejor que ovr).")

# Traducción
import torch
from transformers import MarianMTModel, MarianTokenizer

## 2. Carga y exploración de datos

In [ ]:
df = pd.read_csv("C:/Users/david/OneDrive/Documents/py_r/semillero_patentes/Datos_entrenamiento/ipc_sector.csv")
print("Shape:", df.shape)
print()
df[["Clasificacion", "Sector","Titulo"]].head(8)


In [ ]:
# Distribución de clases
ax = df["Sector"].value_counts().plot(kind="barh", figsize=(8, 3), color="#378ADD")
ax.set_title("Distribución de clases (dataset de entrenamiento)")
ax.set_xlabel("Número de patentes")
for p in ax.patches:
    ax.annotate(f"{int(p.get_width()):,}", (p.get_width() + 200, p.get_y() + 0.3), fontsize=9)
plt.tight_layout()
plt.show()
print(df["Sector"].value_counts())


## 3. Traducción del título: español → inglés

Se usa `Helsinki-NLP/opus-mt-es-en` (MarianMT) en GPU. A diferencia de las APIs
web, aquí sí hay lotes reales: un batch entero pasa por el modelo en una sola
pasada, sin latencia de red ni límite de tasa.

Tres detalles que hacen la diferencia en tiempo:

- **Solo títulos únicos.** Un título repetido entre patentes publicadas y
  concedidas se traduce una vez.
- **Ordenados por longitud.** Agrupar títulos de largo parecido reduce el
  relleno (*padding*) de cada batch. Sin esto, un título largo en un batch de
  cortos obliga a calcular sobre posiciones vacías.
- **Caché en disco.** Al volver a correr el notebook no se retraduce nada.

In [ ]:
MODELO_MT = "Helsinki-NLP/opus-mt-es-en"
CACHE_MT  = "titulos_traducidos.csv"
BATCH     = 64      # bajar a 16 o 32 si aparece CUDA out of memory
MAX_LEN   = 128     # los títulos de patente rara vez pasan de 40 tokens
USAR_FP16 = False   # ver nota abajo

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("ADVERTENCIA: sin GPU la traducción tardará bastante más. "
          "Verificá la instalación de torch con soporte CUDA.")

tokenizador = MarianTokenizer.from_pretrained(MODELO_MT)
modelo_mt = MarianMTModel.from_pretrained(MODELO_MT).to(device)

# fp16 duplica la velocidad, pero MarianMT tiene un problema conocido de
# desbordamiento en media precisión que en algunas GPU devuelve cadenas vacías
# o repeticiones. Por eso está apagado por defecto: activalo y revisá la
# muestra de la celda siguiente antes de confiar en el resultado completo.
if USAR_FP16 and device == "cuda":
    modelo_mt = modelo_mt.half()

modelo_mt.eval()
print("Modelo de traducción cargado.")

In [ ]:
from pathlib import Path


@torch.inference_mode()
def traducir(textos, batch=BATCH, max_len=MAX_LEN):
    """Traduce una lista de cadenas es->en, por lotes y ordenadas por longitud."""
    if not textos:
        return []

    # Se ordena por longitud y al final se restituye el orden original.
    orden = np.argsort([len(t) for t in textos])
    salida = [None] * len(textos)

    for inicio in range(0, len(orden), batch):
        indices = orden[inicio:inicio + batch]
        lote = [textos[i] for i in indices]

        entrada = tokenizador(lote, return_tensors="pt", padding=True,
                              truncation=True, max_length=max_len).to(device)
        # num_beams=1 (greedy) en vez de búsqueda en haz: para títulos cortos
        # la pérdida de calidad es mínima y la velocidad sube 3-4 veces.
        generado = modelo_mt.generate(**entrada, num_beams=1, max_length=max_len)
        traducido = tokenizador.batch_decode(generado, skip_special_tokens=True)

        for i, t in zip(indices, traducido):
            salida[i] = t

        hechos = min(inicio + batch, len(orden))
        print(f"  {hechos:,}/{len(orden):,}", end="\r")

    print()
    return salida


# --- títulos únicos, y solo los que no estén ya traducidos ----------------
df["Titulo"] = df["Titulo"].fillna("").astype(str).str.strip()
titulos = sorted({t for t in df["Titulo"] if t})

if Path(CACHE_MT).exists():
    cache = pd.read_csv(CACHE_MT, encoding="utf-8").dropna()
else:
    cache = pd.DataFrame({"Titulo": pd.Series(dtype=str),
                          "Titulo_en": pd.Series(dtype=str)})

pendientes = sorted(set(titulos) - set(cache["Titulo"]))
print(f"Títulos únicos: {len(titulos):,} | "
      f"en caché: {len(cache):,} | por traducir: {len(pendientes):,}")

if pendientes:
    import time
    t0 = time.time()
    nuevos = pd.DataFrame({"Titulo": pendientes,
                           "Titulo_en": traducir(pendientes)})
    cache = (pd.concat([cache, nuevos], ignore_index=True)
               .drop_duplicates("Titulo", keep="last"))
    cache.to_csv(CACHE_MT, index=False, encoding="utf-8")
    seg = time.time() - t0
    print(f"Traducidos {len(pendientes):,} títulos en {seg/60:.1f} min "
          f"({len(pendientes)/max(seg, 1e-9):.1f} títulos/s)")

# --- mapear de vuelta al DataFrame ---------------------------------------
mapa = dict(zip(cache["Titulo"], cache["Titulo_en"]))
df["Titulo_en"] = df["Titulo"].map(mapa).fillna("")

print(f"\nFilas sin traducción: {(df['Titulo_en'] == '').sum():,}")

In [ ]:
# Revisión manual de una muestra. Si acá aparecen cadenas vacías, texto en
# español sin traducir o repeticiones del tipo "the the the", el modelo se
# desbordó: apagá USAR_FP16 y volvé a correr.
muestra = df.loc[df["Titulo_en"] != "", ["Titulo", "Titulo_en"]].sample(
    min(8, (df["Titulo_en"] != "").sum()), random_state=42)

for original, traducido in muestra.itertuples(index=False):
    print(f"ES: {original[:90]}")
    print(f"EN: {traducido[:90]}")
    print()

vacias = (df["Titulo_en"].str.strip() == "").sum()
print(f"Traducciones vacías: {vacias:,} de {len(df):,}")

In [ ]:
# Liberar la GPU: el modelo de traducción ya no se necesita y ocupa memoria
# que el resto del notebook no va a usar.
del modelo_mt
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"VRAM reservada tras liberar: "
          f"{torch.cuda.memory_reserved(0)/1e9:.2f} GB")

## 4. Preprocesamiento

Dos bloques de texto, cada uno con su propio tratamiento:

- **`texto_ipc`** — se normalizan los separadores para que TF-IDF trate cada
  código IPC como token independiente.
  `A61K 31/19; C07D 417/04` → `A61K 31 19 C07D 417 04`
- **`titulo_en`** — texto natural en inglés, ya traducido arriba.

Van en columnas separadas, no concatenadas en una sola cadena. Concatenarlas
obligaría a usar el mismo vectorizador para códigos y para prosa, cuando cada
uno necesita parámetros distintos (los códigos no tienen *stopwords*, el texto
sí; los códigos raros importan, las palabras raras casi nunca).

In [ ]:
def limpiar_ipc(texto):
    """Normaliza separadores para que cada código IPC sea un token."""
    if pd.isna(texto):
        return ""
    return str(texto).replace(";", " ").replace("/", " ").strip()


df["texto_ipc"] = df["Clasificacion"].apply(limpiar_ipc)
df["titulo_en"] = df["Titulo_en"].fillna("").astype(str)

print("Ejemplo IPC original: ", df["Clasificacion"].iloc[1])
print("Ejemplo IPC procesado:", df["texto_ipc"].iloc[1])
print("Ejemplo título:       ", df["titulo_en"].iloc[1][:80])
print()
print(f"Registros sin IPC:    {(df['texto_ipc'] == '').sum():,}")
print(f"Registros sin título: {(df['titulo_en'] == '').sum():,}")

## 5. División train / test (80 / 20)

**Cambio respecto a la versión anterior.** Antes había un `df.dropna()` sin
argumentos, que borraba cualquier fila con un nulo en *cualquier* columna. Con
la columna de traducción agregada eso eliminaría también las filas cuyo título
no se pudo traducir, aunque su IPC estuviera perfecto. Ahora se exige solo lo
que el modelo realmente necesita: la etiqueta, y al menos uno de los dos
bloques de texto.

In [ ]:
COLS_TEXTO = ["texto_ipc", "titulo_en"]

antes = len(df)
df = df[df["Sector"].notna()].copy()
df[COLS_TEXTO] = df[COLS_TEXTO].fillna("")
# Una fila sin IPC y sin título no aporta ninguna señal.
df = df[(df["texto_ipc"] != "") | (df["titulo_en"] != "")]
print(f"Filas descartadas: {antes - len(df):,} de {antes:,}")

X = df[COLS_TEXTO]          # DataFrame, no Series: el pipeline usa dos columnas
y = df["Sector"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\nTrain: {len(X_train):,} registros")
print(f"Test:  {len(X_test):,} registros")
print()
print("Distribución train:")
print(y_train.value_counts())

## 6. Pipeline: ColumnTransformer (IPC + título) + Regresión Logística

**¿Por qué este modelo?**
- `Clasificacion` es texto estructurado con vocabulario finito (~25k tokens IPC únicos)
- TF-IDF captura la relevancia relativa de cada código por clase
- Regresión Logística multiclase es interpretable y muy eficiente en matrices sparse
- `class_weight='balanced'` corrige el desbalance (Química 58% vs Electricidad 7%)

**Por qué agregar el título.** El código IPC es asignado por un examinador y ya
condensa buena parte de la información del sector, así que el margen de mejora
es acotado. Donde el título rinde es en las filas sin IPC o con un IPC genérico
poco informativo: ahí antes no había nada que clasificar y ahora sí.

Cada bloque lleva su propio vectorizador. Para el título se usa `min_df=3`
—palabras que aparecen una o dos veces en todo el corpus son ruido— y
*stopwords* en inglés.

In [ ]:
features = ColumnTransformer(
    transformers=[
        ("ipc", TfidfVectorizer(
            ngram_range=(1, 2),   # unigramas y bigramas de códigos IPC
            min_df=2,             # ignorar códigos que aparecen menos de 2 veces
            sublinear_tf=True,    # log(tf+1): reduce el efecto de frecuencias altas
        ), "texto_ipc"),
        ("titulo", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,             # más estricto: el texto natural tiene cola larga
            sublinear_tf=True,
            stop_words="english",
            strip_accents="unicode",
            lowercase=True,
        ), "titulo_en"),
    ],
    # Subir el peso del título (p. ej. 0.5 -> 1.5) si al comparar abajo resulta
    # que aporta más de lo que este valor le permite.
    transformer_weights={"ipc": 1.0, "titulo": 1.0},
)

pipeline = Pipeline([
    ("features", features),
    # class_weight="balanced" corrige el desbalance; el resto de parámetros
    # está en PARAMS_LR (celda de librerías) para no duplicarlos.
    ("clf", LogisticRegression(**PARAMS_LR)),
])

pipeline.fit(X_train, y_train)
print("Modelo entrenado correctamente.")
print(f"Dimensión del espacio de features: "
      f"{pipeline.named_steps['features'].transform(X_train.head(1)).shape[1]:,}")

### ¿Cuánto aporta realmente el título?

Un modelo que suma features casi siempre sube algo en entrenamiento. La
pregunta que importa es si sube en test, y cuánto. Esta celda entrena las tres
variantes sobre la misma partición para que la comparación sea justa.

In [ ]:
def construir(cols, pesos=None):
    """Arma el mismo pipeline restringido a un subconjunto de bloques."""
    disponibles = {
        "texto_ipc": ("ipc", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                             sublinear_tf=True)),
        "titulo_en": ("titulo", TfidfVectorizer(ngram_range=(1, 2), min_df=3,
                                                sublinear_tf=True,
                                                stop_words="english",
                                                strip_accents="unicode")),
    }
    trs = [(disponibles[c][0], disponibles[c][1], c) for c in cols]
    return Pipeline([
        ("features", ColumnTransformer(trs, transformer_weights=pesos)),
        ("clf", LogisticRegression(**PARAMS_LR)),
    ])


comparacion = []
for etiqueta, cols in [("Solo IPC", ["texto_ipc"]),
                       ("Solo título (en)", ["titulo_en"]),
                       ("IPC + título", ["texto_ipc", "titulo_en"])]:
    m = construir(cols).fit(X_train, y_train)
    pred = m.predict(X_test)
    comparacion.append({
        "Modelo": etiqueta,
        "Accuracy test": round(accuracy_score(y_test, pred), 4),
    })

# Caso que justifica el cambio: filas sin IPC, donde el modelo viejo no podía
# predecir nada. Si no hay ninguna en el test, el resultado sale vacío.
sin_ipc = X_test["texto_ipc"] == ""
if sin_ipc.sum() > 0:
    m_full = construir(["texto_ipc", "titulo_en"]).fit(X_train, y_train)
    acc_sin_ipc = accuracy_score(y_test[sin_ipc], m_full.predict(X_test[sin_ipc]))
    print(f"Filas de test sin IPC: {sin_ipc.sum():,} — "
          f"accuracy usando solo el título: {acc_sin_ipc:.4f}\n")
else:
    print("No hay filas sin IPC en el conjunto de test.\n")

pd.DataFrame(comparacion).set_index("Modelo")

## 7. Evaluación en conjunto de test

In [ ]:
y_pred = pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))


In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=pipeline.classes_)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=pipeline.classes_,
            yticklabels=pipeline.classes_,
            ax=ax, linewidths=0.5)
ax.set_title("Matriz de confusión (normalizada) — conjunto test")
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 8. Validación cruzada (5-fold)

In [ ]:
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring="accuracy", n_jobs=-1)
print(f"Accuracy CV 5-fold:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Por fold: {[round(s, 4) for s in cv_scores]}")


## 9. Features más importantes por sector

`ColumnTransformer` prefija los nombres con el bloque del que vienen
(`ipc__` / `titulo__`), así que en el mismo gráfico se ve qué pesa más: un
código o una palabra del título.

In [ ]:
feature_names = pipeline.named_steps["features"].get_feature_names_out()
coef = pipeline.named_steps["clf"].coef_
classes = pipeline.classes_

fig, axes = plt.subplots(1, len(classes), figsize=(22, 5), sharey=False)
fig.suptitle("Top 10 features más predictivas por sector "
             "(ipc__ = código, titulo__ = palabra del título)",
             fontsize=13, fontweight="500")

for i, (cls, ax) in enumerate(zip(classes, axes)):
    top_idx = np.argsort(coef[i])[-10:][::-1]
    top_features = feature_names[top_idx]
    top_coefs = coef[i][top_idx]
    colores = ["#378ADD" if f.startswith("ipc__") else "#EF9F27"
               for f in top_features]
    ax.barh(top_features[::-1], top_coefs[::-1], color=colores[::-1])
    ax.set_title(cls, fontsize=10)
    ax.tick_params(axis="y", labelsize=7)
    ax.set_xlabel("Coeficiente")

plt.tight_layout()
plt.show()

# Reparto global: ¿cuánta masa de coeficientes viene de cada bloque?
es_ipc = np.array([f.startswith("ipc__") for f in feature_names])
peso_ipc = np.abs(coef[:, es_ipc]).sum()
peso_tit = np.abs(coef[:, ~es_ipc]).sum()
print(f"Masa de coeficientes — IPC: {peso_ipc/(peso_ipc+peso_tit):.1%}  ·  "
      f"Título: {peso_tit/(peso_ipc+peso_tit):.1%}")

In [ ]:
import joblib

# Se guarda el pipeline completo: incluye los DOS vectorizadores, así que al
# cargarlo en otro script espera un DataFrame con las columnas
# ["texto_ipc", "titulo_en"], no una lista de cadenas.
joblib.dump(pipeline, "modelo_clf_patentes.pkl")
print("Guardado: modelo_clf_patentes.pkl")
print("Entrada esperada: DataFrame con columnas", COLS_TEXTO)

## 10. Aplicar el modelo al nuevo dataset

### Compatibilidad con el modelo

| Aspecto | Resultado |
|---|---|
| Formato IPC | Idéntico (`A61K 31/19; C07D 417/04`) |
| Overlap de códigos únicos | **96.7%** del nuevo dataset está visto en entrenamiento |
| Códigos no vistos | 614 de 18 802 (3.3%) — manejados por TF-IDF con peso 0 |
| Registros con IPC nulo | 3 189 (8.2%) — ahora **sí clasificables** si tienen título |

**El título del dataset nuevo NO se traduce: ya está en inglés.** Ese es
justamente el idioma al que se llevó el entrenamiento. La columna `Title` se
mapea directamente a `titulo_en`.

**Advertencia sobre el desfase de dominio.** Los títulos de entrenamiento son
traducción automática de textos en español; los de aplicación son texto
original en inglés (o traducidos por la OMPI con otro sistema). La traducción
automática deja huellas de estilo —giros literales, vocabulario más restringido—
que el modelo puede aprender como si fueran señal de sector. Es un riesgo real,
y la mejor defensa es la comparación de accuracy de la sección 6: si el título
aporta poco en test, aporta todavía menos fuera de él.

In [ ]:
df_new = pd.read_excel("C:/Users/david/OneDrive/Documents/py_r/semillero_patentes/Limpieza_datos_OMPI/Patentes_OMPI/merged_patents.xlsx")
print("Shape:", df_new.shape)
print("Columnas:", df_new.columns.tolist())
print()

# Verificación explícita: si el archivo cambia de nombres de columna, es mejor
# enterarse acá que tres celdas más abajo con un KeyError.
faltan = [c for c in ["IPC", "Title"] if c not in df_new.columns]
if faltan:
    raise KeyError(f"El dataset nuevo no tiene las columnas {faltan}. "
                   f"Disponibles: {df_new.columns.tolist()}")

print(f"Registros con IPC nulo:    {df_new['IPC'].isnull().sum():,} "
      f"({df_new['IPC'].isnull().mean()*100:.1f}%)")
print(f"Registros con Title nulo:  {df_new['Title'].isnull().sum():,} "
      f"({df_new['Title'].isnull().mean()*100:.1f}%)")
df_new.head(4)

In [ ]:
# Mismo preprocesamiento que en entrenamiento, columna por columna.
df_new["texto_ipc"] = df_new["IPC"].apply(limpiar_ipc)
df_new["titulo_en"] = (df_new["Title"].fillna("").astype(str).str.strip())

# Ahora basta con tener UNO de los dos bloques. Antes, una patente sin IPC se
# descartaba aunque tuviera título; con el título como feature ya es
# clasificable, que es la ganancia concreta de este cambio.
tiene_señal = (df_new["texto_ipc"] != "") | (df_new["titulo_en"] != "")

df_clasificable   = df_new[tiene_señal].copy()
df_no_clasificable = df_new[~tiene_señal].copy()

solo_titulo = ((df_clasificable["texto_ipc"] == "")
               & (df_clasificable["titulo_en"] != "")).sum()

print(f"Registros a clasificar:            {len(df_clasificable):,}")
print(f"   de los cuales, solo por título: {solo_titulo:,}")
print(f"Sin IPC ni título (no clasificable): {len(df_no_clasificable):,}")

In [ ]:
# El pipeline espera un DataFrame con las dos columnas, no una Series.
X_new = df_clasificable[COLS_TEXTO]

df_clasificable["Sector_predicho"] = pipeline.predict(X_new)
proba = pipeline.predict_proba(X_new)
df_clasificable["Confianza"] = proba.max(axis=1).round(4)


def nivel_confianza(p):
    if p >= 0.80:
        return "Alta"
    elif p >= 0.50:
        return "Media"
    return "Baja"


df_clasificable["Nivel_confianza"] = df_clasificable["Confianza"].apply(nivel_confianza)

# Marca de procedencia: útil para filtrar después. Una predicción hecha solo
# con el título no merece la misma confianza que una respaldada por el IPC,
# aunque el número de probabilidad se parezca.
df_clasificable["Fuente"] = np.where(
    df_clasificable["texto_ipc"] == "", "Solo título",
    np.where(df_clasificable["titulo_en"] == "", "Solo IPC", "IPC + título"))

# Vista previa, tolerante a que falte alguna columna en el archivo.
cols_vista = [c for c in ["Application_Number", "Title", "IPC", "Sector_predicho",
                          "Confianza", "Nivel_confianza", "Fuente"]
              if c in df_clasificable.columns]
df_clasificable[cols_vista].head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df_clasificable["Sector_predicho"].value_counts().plot(
    kind="barh", ax=axes[0], color="#378ADD")
axes[0].set_title("Distribución de sectores predichos")
axes[0].set_xlabel("Número de patentes")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_width()):,}",
                     (p.get_width() + 50, p.get_y() + 0.25), fontsize=9)

orden_conf = [c for c in ["Alta", "Media", "Baja"]
              if c in df_clasificable["Nivel_confianza"].unique()]
colores = {"Alta": "#1D9E75", "Media": "#EF9F27", "Baja": "#E24B4A"}
df_clasificable["Nivel_confianza"].value_counts().reindex(orden_conf).plot(
    kind="bar", ax=axes[1], color=[colores[c] for c in orden_conf])
axes[1].set_title("Nivel de confianza de las predicciones")
axes[1].set_ylabel("Número de patentes")
axes[1].tick_params(axis="x", rotation=0)
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + 0.1, p.get_height() + 50), fontsize=9)

df_clasificable["Fuente"].value_counts().plot(
    kind="bar", ax=axes[2], color="#7C5CBF")
axes[2].set_title("Origen de la señal usada")
axes[2].set_ylabel("Número de patentes")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print("\nConfianza media según la señal disponible:")
print(df_clasificable.groupby("Fuente")["Confianza"].agg(["count", "mean"]).round(3))

## 11. Exportar resultados

In [ ]:
# Las filas sin IPC ni título se marcan explícitamente en vez de perderse.
if len(df_no_clasificable) > 0:
    df_no_clasificable["Sector_predicho"] = "Sin IPC ni título - no clasificable"
    df_no_clasificable["Confianza"] = np.nan
    df_no_clasificable["Nivel_confianza"] = "No clasificable"
    df_no_clasificable["Fuente"] = "Ninguna"

df_resultado = pd.concat([df_clasificable, df_no_clasificable], ignore_index=True)

# errors="ignore" por si alguna de las auxiliares ya no está: el export no
# debería caerse por una columna de trabajo.
df_resultado = df_resultado.drop(columns=COLS_TEXTO, errors="ignore")

df_resultado.to_csv("combined_dataset_clasificado.csv", index=False,
                    encoding="utf-8-sig")
print("Archivo guardado: combined_dataset_clasificado.csv")
print(f"Shape final: {df_resultado.shape}")
assert len(df_resultado) == len(df_new), "Se perdieron filas en el proceso"

cols_final = [c for c in ["Application_Number", "Title", "IPC", "Sector_predicho",
                          "Confianza", "Nivel_confianza", "Fuente"]
              if c in df_resultado.columns]
df_resultado[cols_final].head(10)